In [ ]:
import pandas as pd
companies = pd.read_csv("../../nyse_nasdaq_companies_all.csv")

companies["company_id"] = companies["company_id"].astype(str).str.strip()

ticker_map = dict(zip(companies["company_id"], companies["primary_ticker"]))

In [2]:
import ast
import pandas as pd

events = pd.read_csv("company_event_high_confidence_for_graph.csv")

def parse_list_column(x):
    if isinstance(x, list):
        return x
    try:
        value = ast.literal_eval(str(x))
        if isinstance(value, list):
            return value
        return []
    except Exception:
        return []

events["linked_company_ids"] = events["linked_company_ids"].apply(parse_list_column)
events["linked_companies"] = events["linked_companies"].apply(parse_list_column)

events_exploded = events.explode(["linked_company_ids", "linked_companies"]).copy()

events_exploded["company_id"] = events_exploded["linked_company_ids"].astype(str).str.strip()
events_exploded["company"] = events_exploded["linked_companies"]

events_exploded["ticker"] = events_exploded["company_id"].map(ticker_map)

events_exploded = events_exploded[
    events_exploded["ticker"].notna()
].copy()

events_exploded[["date", "company", "ticker", "event_type", "sentence"]].head()

,date,company,ticker,event_type,sentence
3,2018-04-10,Shell,RDS.B,ipo_listing,"The public offering of the company, which sell..."
4,2018-05-04,Alaska Air Group,ALK,dividend,The board of directors of Alaska Air Group (NY...
5,2018-05-08,BioCryst Pharmaceuticals,BCRX,earnings_results,"RESEARCH TRIANGLE PARK, N.C., May 08, 2018 (GL..."
6,2018-02-05,"Netflix, Inc.",NFLX,acquisition_merger,The move was made that much more surprising by...
6,2018-02-05,Viacom,VIA,acquisition_merger,The move was made that much more surprising by...


In [19]:
import yfinance as yf
import pandas as pd
import numpy as np

events_exploded["date"] = pd.to_datetime(events_exploded["date"])

tickers = sorted(events_exploded["ticker"].dropna().unique())

start_date = events_exploded["date"].min() - pd.Timedelta(days=10)
end_date = events_exploded["date"].max() + pd.Timedelta(days=15)

prices = yf.download(
    tickers=tickers,
    start=start_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    auto_adjust=True,
    progress=True,
    group_by="ticker"
)

[                       0%                       ]  2 of 1486 completed$ROSG: possibly delisted; no price data found  (1d 2017-12-23 -> 2018-09-08)
[                       1%                       ]  8 of 1486 completed$AERI: possibly delisted; no timezone found
$USG: possibly delisted; no price data found  (1d 2017-12-23 -> 2018-09-08) (Yahoo error = "Data doesn't exist for startDate = 1514005200, endDate = 1536379200")
[                       1%                       ]  10 of 1486 completed$ZUO: possibly delisted; no timezone found
[                       1%                       ]  11 of 1486 completed$ENV: possibly delisted; no timezone found
[                       1%                       ]  12 of 1486 completed$DISCK: possibly delisted; no timezone found
$K: possibly delisted; no timezone found
[                       1%                       ]  22 of 1486 completed$CBST: possibly delisted; no price data found  (1d 2017-12-23 -> 2018-09-08)
[*                      2%            

In [20]:
def get_adjusted_close(prices, ticker):
    """
    Return adjusted close series for one ticker from yfinance output.
    Works for multi-ticker and single-ticker downloads.
    """
    try:
        if isinstance(prices.columns, pd.MultiIndex):
            if ticker in prices.columns.get_level_values(0):
                return prices[ticker]["Close"].dropna()
        else:
            return prices["Close"].dropna()
    except Exception:
        return pd.Series(dtype=float)

    return pd.Series(dtype=float)


def get_event_window_return(price_series, event_date, window_days=1):
    """
    Compute return from the first trading day on/after event_date
    to window_days trading days after.
    """
    if price_series is None or len(price_series) == 0:
        return np.nan

    # Force Series, not DataFrame
    if isinstance(price_series, pd.DataFrame):
        if price_series.shape[1] == 1:
            price_series = price_series.iloc[:, 0]
        else:
            return np.nan

    price_series = price_series.dropna().sort_index()
    price_series.index = pd.to_datetime(price_series.index).tz_localize(None)

    event_date = pd.to_datetime(event_date)
    if getattr(event_date, "tzinfo", None) is not None:
        event_date = event_date.tz_localize(None)

    future_prices = price_series[price_series.index >= event_date]

    if len(future_prices) <= window_days:
        return np.nan

    p0 = float(future_prices.iloc[0])
    p1 = float(future_prices.iloc[window_days])

    if p0 == 0 or pd.isna(p0) or pd.isna(p1):
        return np.nan

    return (p1 - p0) / p0

In [21]:
returns = []

for _, row in events_exploded.iterrows():
    ticker = row["ticker"]
    event_date = row["date"]

    price_series = get_adjusted_close(prices, ticker)

    returns.append({
        "article_id": row["article_id"],
        "date": event_date,
        "company_id": row["company_id"],
        "company": row["company"],
        "ticker": ticker,
        "event_type": row["event_type"],
        "trigger": row["trigger"],
        "return_1d": get_event_window_return(price_series, event_date, window_days=1),
        "return_3d": get_event_window_return(price_series, event_date, window_days=3),
        "return_5d": get_event_window_return(price_series, event_date, window_days=5),
        "sentence": row["sentence"]
    })

event_returns = pd.DataFrame(returns)

event_returns.to_csv("event_stock_returns.csv", index=False)

event_returns.head()

,article_id,date,company_id,company,ticker,event_type,trigger,return_1d,return_3d,return_5d,sentence
0,2,2018-04-10,Q154950,Shell,RDS.B,ipo_listing,public offering,NaN,NaN,NaN,"The public offering of the company, which sell..."
1,6,2018-05-04,Q4033665,Alaska Air Group,ALK,dividend,declared a regular quarterly cash dividend,-0.018312,-0.038376,-0.036147,The board of directors of Alaska Air Group (NY...
2,7,2018-05-08,Q4914611,BioCryst Pharmaceuticals,BCRX,earnings_results,financial results,0.059642,0.093439,0.174950,"RESEARCH TRIANGLE PARK, N.C., May 08, 2018 (GL..."
3,9,2018-02-05,Q116452644,"Netflix, Inc.",NFLX,acquisition_merger,acquisition,0.045072,-0.016361,0.014513,The move was made that much more surprising by...
4,9,2018-02-05,Q214346,Viacom,VIA,acquisition_merger,acquisition,NaN,NaN,NaN,The move was made that much more surprising by...


In [ ]:
spy = yf.download(
    "SPY",
    start=start_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    auto_adjust=True,
    progress=False
)

if isinstance(spy.columns, pd.MultiIndex):
    spy_close = spy[("Close", "SPY")].dropna()
else:
    spy_close = spy["Close"].dropna()

spy_close.index = pd.to_datetime(spy_close.index).tz_localize(None)

spy_close = spy["Close"].dropna()

event_returns["market_return_1d"] = event_returns["date"].apply(
    lambda d: get_event_window_return(spy_close, d, 1)
)

event_returns["market_return_3d"] = event_returns["date"].apply(
    lambda d: get_event_window_return(spy_close, d, 3)
)

event_returns["market_return_5d"] = event_returns["date"].apply(
    lambda d: get_event_window_return(spy_close, d, 5)
)

event_returns["abnormal_return_1d"] = (
    event_returns["return_1d"] - event_returns["market_return_1d"]
)

event_returns["abnormal_return_3d"] = (
    event_returns["return_3d"] - event_returns["market_return_3d"]
)

event_returns["abnormal_return_5d"] = (
    event_returns["return_5d"] - event_returns["market_return_5d"]
)

event_returns.to_csv("event_stock_abnormal_returns.csv", index=False)

In [3]:
event_returns=pd.read_csv("event_stock_abnormal_returns.csv")

In [5]:
event_return_summary = event_returns.groupby("event_type").agg(
    n_events=("article_id", "count"),
    mean_return_1d=("return_1d", "mean"),
    median_return_1d=("return_1d", "median"),
    mean_return_3d=("return_3d", "mean"),
    median_return_3d=("return_3d", "median"),
    mean_abnormal_return_3d=("abnormal_return_3d", "mean"),
    median_abnormal_return_3d=("abnormal_return_3d", "median"),
    volatility_3d=("return_3d", "std")
).reset_index()

event_return_summary.sort_values("mean_abnormal_return_3d")

,event_type,n_events,mean_return_1d,median_return_1d,mean_return_3d,median_return_3d,mean_abnormal_return_3d,median_abnormal_return_3d,volatility_3d
0,acquisition_merger,2489,-0.001394,0.000000,-0.002309,-0.000678,-0.002697,-0.001171,0.051605
7,partnership_contract,622,0.000746,0.000000,-0.002727,-0.002359,-0.001406,-0.001813,0.047416
5,lawsuit_legal,1406,-0.001399,0.000000,-0.001025,-0.002423,-0.000269,-0.000613,0.042902
8,regulatory_approval,87,0.000963,0.005514,0.005853,0.004598,0.000571,0.001983,0.040389
4,ipo_listing,551,-0.004293,-0.001308,-0.003646,-0.001367,0.000816,0.002095,0.052091
2,downgrade,86,-0.007002,-0.005091,0.001912,-0.000538,0.001902,-0.004437,0.047769
3,earnings_results,2007,0.003078,0.001417,0.004756,0.002814,0.002595,0.000569,0.055842
6,layoffs_restructuring,309,0.002697,0.000858,0.003327,0.000000,0.002965,-0.002593,0.064553
1,dividend,391,0.006262,0.002789,0.007531,0.003374,0.006214,0.002383,0.044890


In [ ]:
NEGATIVE_EVENTS = {
    "lawsuit_legal",
    "downgrade",
    "layoffs_restructuring"
}

event_returns["negative_event"] = event_returns["event_type"].isin(NEGATIVE_EVENTS)

negative_returns = event_returns[
    event_returns["negative_event"]
]["abnormal_return_3d"].dropna()

other_returns = event_returns[
    ~event_returns["negative_event"]
]["abnormal_return_3d"].dropna()

In [6]:
NEGATIVE_EVENTS = {
    "lawsuit_legal",
    "downgrade",
    "layoffs_restructuring"
}

event_returns["negative_event"] = event_returns["event_type"].isin(NEGATIVE_EVENTS)

negative_returns = event_returns[
    event_returns["negative_event"]
]["abnormal_return_1d"].dropna()

other_returns = event_returns[
    ~event_returns["negative_event"]
]["abnormal_return_1d"].dropna()

In [7]:
from scipy.stats import mannwhitneyu

stat, p = mannwhitneyu(
    negative_returns,
    other_returns,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", stat)
print("p-value:", p)
print("Median negative event abnormal return:", negative_returns.median())
print("Median other event abnormal return:", other_returns.median())

Mann-Whitney U statistic: 2861264.0
p-value: 0.019488037715291086
Median negative event abnormal return: -0.00029985981765
Median other event abnormal return: 0.0001100042175749


In [26]:
POSITIVE_EARNINGS_TERMS = [
    "beat", "beats", "higher-than-expected", "better-than-expected",
    "profit rose", "revenue increased", "sales increased",
    "record revenue", "strong results"
]

NEGATIVE_EARNINGS_TERMS = [
    "miss", "missed", "lower-than-expected", "weaker-than-expected",
    "net loss", "loss widened", "revenue decreased",
    "sales decreased", "profit fell"
]

def classify_earnings_tone(sentence):
    s = str(sentence).lower()

    if any(term in s for term in POSITIVE_EARNINGS_TERMS):
        return "positive"

    if any(term in s for term in NEGATIVE_EARNINGS_TERMS):
        return "negative"

    return "neutral_or_unclear"

event_returns["earnings_tone"] = np.where(
    event_returns["event_type"] == "earnings_results",
    event_returns["sentence"].apply(classify_earnings_tone),
    None
)

event_returns[event_returns["event_type"] == "earnings_results"][
    ["company", "event_type", "earnings_tone", "abnormal_return_3d", "sentence"]
].head(20)

,company,event_type,earnings_tone,abnormal_return_3d,sentence
2,BioCryst Pharmaceuticals,earnings_results,neutral_or_unclear,0.071223,"RESEARCH TRIANGLE PARK, N.C., May 08, 2018 (GL..."
5,ArcelorMittal,earnings_results,positive,0.015953,"May 11, 2018 / 5:26 AM / Updated 16 minutes ag..."
6,Unisys,earnings_results,neutral_or_unclear,-0.020960,"BLUE BELL, Pa., April 18, 2018 /PRNewswire/ --..."
12,Colliers International,earnings_results,neutral_or_unclear,-0.005975,Colliers International Group Inc : * QTRLY ADJ...
13,Luby's,earnings_results,neutral_or_unclear,NaN,"HOUSTON, April 23, 2018 /PRNewswire/ -- Luby's..."
21,"Nasdaq, Inc.",earnings_results,neutral_or_unclear,0.014043,"MOBILE, Ala., April 19, 2018 (GLOBE NEWSWIRE) ..."
27,"Nasdaq, Inc.",earnings_results,neutral_or_unclear,0.019250,"GUANGZHOU, China, May 23, 2018 (GLOBE NEWSWIRE..."
29,People's United Financial,earnings_results,neutral_or_unclear,NaN,"People's United Financial, Inc.\nFINANCIAL HIG..."
33,"Nasdaq, Inc.",earnings_results,neutral_or_unclear,0.002456,"Corium International, Inc. (NASDAQ:CORI) today..."
35,Mylan,earnings_results,neutral_or_unclear,NaN,"EARNINGS IN LINE, BUT SALES DIP\nMylan said it..."


In [27]:
earnings_only = event_returns[
    event_returns["event_type"] == "earnings_results"
].copy()

earnings_summary = earnings_only.groupby("earnings_tone").agg(
    n_events=("article_id", "count"),
    mean_abnormal_return_3d=("abnormal_return_3d", "mean"),
    median_abnormal_return_3d=("abnormal_return_3d", "median"),
    volatility_3d=("return_3d", "std")
).reset_index()

earnings_summary

,earnings_tone,n_events,mean_abnormal_return_3d,median_abnormal_return_3d,volatility_3d
0,negative,135,-0.004488,0.000177,0.046870
1,neutral_or_unclear,1819,0.003143,0.000919,0.056862
2,positive,53,-0.000550,-0.000178,0.032852
